ПАЙПЛАЙН коротко: слово в контексте → эмбеддинг → эмбеддинг словарной дефиниции → эмбеддинг определения семантического класса → семантический класс

# Пайплайн подробно

# 1. Грузим эмбеддинги и нужные модули

1) импортируем нужные библиотеки
2) загружаем предобученную модель НКРЯ основной корпус (main_POS_V300_W5_C5)
3) проверяем формат и размерность эмбеддингов
4) функция для подсчета вектора предложения

---
# 2. Парсим conllu с обучающими данными

1) загружаем обучающий корпус в формате CoNLL-U
2) Приводим токены CoNLL-U к формату Word2Vec НКРЯ
	1) получаем lemma и UPOS
	2) преобразуем UPOS в обозначение НКРЯ
	3) формируем token = lemma + "_" + POS
	4) проверяем наличие token в словаре Word2Vec

---
# 3. Подготавливаем дефиниции

1) загружаем датасет с толкованиями: russian-reverse-dictionary-train-data
2) формируем структуру: слово → список его дефиниций
3) переводим дефиниции в векторный формат
	1) получаем embeddings всех дефиниций
	2) сохраняем соответствие: (слово, дефиниция) → embedding
	3) сохраняем рассчитанные embeddings на Google Drive, чтобы не пересчитывать их при каждом запуске

---
# 4. Подготавливаем теги

1) загружаем список семантических тегов в словарь: тег → описание/толкование тега
2) ??? при необходимости очищаем и нормализуем тексты тегов
3) Переводим теги в векторный формат
	1) получаем тексты всех тегов
	2) рассчитываем embedding для каждого тега
	3) сохраняем соответствие: тег → embedding на Google Drive, чтобы не рассчитывать их повторно

---

# 5. Обрабатываем слова

1) из набора текстов извлекаем предложения
2) в предложении извлекаем информацию о каждом токене: ID, FORM, LEMMA, UPOS, FEATS и т. д.
3) для предложения считаем его вектор (это будет вектор контекста для значимых токенов)
3) идем по каждому токену; проверяем на то, является ли токен значимым словом: существительные, глаголы, прилагательные и тд); для него будем автоматически присваивать теги
	1) сохраняем правильные семантические теги
	2) получаем лемму слова
	3) находим все дефиниции этой леммы
		1) если дефиниций нет, фиксируем —
	4) берем вектор предложения (считали выше) и сравниваем его с векторами дефиниций; выбираем наиболее близкую дефиницию
	5) сравниваем вектор наиболее близкой дефиниции с векторами описаний тегов; отсекаем набор тегов по threshold
	6) сохраняем результат
		1) слово
		2) лемма
		3) контекст
		4) выбранная дефиниция
		5) выбранный семантический тег
		6) значение similarity
		7) эталонный набор тегов

		типа такого:

	```
	{
    "form": "мира",
    "lemma": "мир",
    "context": "После войны наступил долгожданный мир.",
    "gold_tags": ["abstr"],
    "definition": "состояние отсутствия войны",
    "definition_similarity": 0.73,
    "predicted_tags": ["abstr", "state"],
    "tag_similarities": {
        "abstr": 0.81,
        "object": 0.32,
        "state": 0.76
    }
	```


}

---

# 6. Оцениваем результаты…

сравниваем автоматически выбранное значение с эталонной разметкой
https://medium.com/apprentice-journal/evaluating-multi-class-classifiers-12b2946e755b

рассчитываем:
1) Accuracy
2) Precision
3) Recall
4) F1

отдельно анализируем:
- многозначные слова
- однозначные слова
- случаи отсутствия дефиниции
- случаи низкой similarity



---



# 1 Грузим эмбеддинги и нужные библиотеки

In [1]:
!pip install torch transformers scikit-learn sentence-transformers


Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 10, in <module>
    sys.exit(main())
             ~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/main.py", line 78, in main
    command = create_command(cmd_name, isolated=("--isolated" in cmd_args))
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/commands/__init__.py", line 114, in create_command
    module = importlib.import_module(module_path)
  File "/usr/lib/python3.13/importlib/__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1395, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/Аспирантура/main_POS_V300_W5_C5.model"


In [ ]:
!pip install gensim


In [ ]:
from gensim.models import Word2Vec
model = Word2Vec.load(MODEL_PATH)


### 1.1 Смотрим на модель

In [ ]:
print(model)


Word2Vec<vocab=256947, vector_size=300, alpha=0.025>


In [ ]:
print(model.wv['животное_S'])


[ 1.3416992   0.32486817 -1.4224169   1.3692266  -0.8321114  -1.3630662
  2.1358366  -0.46853593 -0.82078826 -0.06594151 -1.5427055   0.79714745
 -1.3727516   1.4832953   0.829384   -1.1181526   1.9436599   0.03120681
  1.4723561   1.2797338  -1.8804896   0.63048893  1.0314163  -0.59077984
 -0.6947635  -2.5529957  -2.1255815  -1.8356557  -1.5212271  -0.6488539
  0.03841949 -0.7256603   0.24212977  1.0880939   0.47263035  2.0024395
  0.8916925   0.08779316  1.3082622  -1.47039     0.9235288  -0.8781539
  0.8473487  -1.7029902  -0.6209238   0.35167     0.376733   -0.37658483
 -0.7016667   1.3130687   1.1191815  -0.06338452 -0.47453216  0.68348914
 -0.03078272 -0.4422737   1.1898123  -1.9914191   3.0459118   1.7767409
  0.5900429  -0.98016655 -0.87719345 -3.015989   -0.03799308 -0.35175368
  0.39825317  0.320176   -0.16223514 -0.90062076  0.489398    0.52655333
  1.842554   -0.5093195   0.5215699   0.7103413   0.9285111   1.5784369
  1.5521775  -0.3659774   0.5430515  -0.63560516  0.22407

In [ ]:
print(model.wv.most_similar(f'лингвистика_S', topn=10))


[('литературоведение_S', 0.7757636308670044), ('социология_S', 0.7748944163322449), ('естествознание_S', 0.7439813017845154), ('филология_S', 0.7432325482368469), ('языкознание_S', 0.733309268951416), ('педагогика_S', 0.7182799577713013), ('семиотика_S', 0.7047955989837646), ('антропология_S', 0.7024316191673279), ('политология_S', 0.7014004588127136), ('психиатрия_S', 0.6922852993011475)]


In [ ]:
print(model.wv.similarity('оружие_S', 'инструмент_S'))
print(model.wv.similarity('оружие_S', 'орудие_S'))
print(model.wv.similarity('орудие_S', 'инструмент_S'))


0.44086814
0.545498
0.46618852


In [ ]:
if 'инструмент_S' in model.wv.key_to_index:
  print('yes')


yes




---



# 2 Обрабатываем конлю

## 2.1 Функции для парсинга

In [ ]:
!pip install conllu


### 2.1.1 Вытаскиваем семантические теги

In [ ]:
import re
SEM_PATTERN = re.compile(r"sem=\[([^\]]+)\]")

def extract_sem_tags(misc): # вытаскиваем семантические теги
    if misc is None:
        return []

    # случай: conllu вернул dict
    if isinstance(misc, dict):
        sem_val = misc.get("sem")
        if not sem_val:
            return []
        sem_val = sem_val.strip("[]")
        return [t.strip() for t in sem_val.split(",")]

    # случай: строка
    match = SEM_PATTERN.search(str(misc))
    if not match:
        return []

    return [t.strip() for t in match.group(1).split(",")]


### 2.1.2 Приводим токены CoNLL-U к формату Word2Vec НКРЯ (чтобы потом считать векторы слов и предложений)

In [ ]:
pos_map = {
    "NOUN":  "S",       # существительное
    "ADJ":   "A",       # прилагательное
    "VERB":  "V",       # глагол
    "ADV":   "ADV",     # наречие
    "PRON":  "SPRO",    # местоимение
    "ADVPRO": "ADV",    # местоименное наречие
    "NUM":   "NUM",     # числительное
    "PROPN": "S",       # имя собственное → существительное

    "ADP":   "PR",      # предлог
    "CCONJ": "CONJ",    # сочинительный союз
    "SCONJ": "CONJ",    # подчинительный союз
    "PART":  "PART",    # частица
    "INTJ":  "INTJ",    # междометие

    "DET":   "APRO",    # определительное/местоименное прилагательное
    "AUX":   "V",       # вспомогательный глагол
    "SYM":   "SYM",     # символ
    "X":     "X",       # другое / неизвестное
    }

def conllu_to_w2v_format(lemma, upos): # POS MyStem → POS UD
    pos = pos_map.get(upos)

    if pos is None:
        return lemma

    return f"{lemma}_{pos}"

# TODO: посмотреть отдельные слова, например, здесь ADVPRO не переводится в ADV
# [w for w in wv.key_to_index if w.startswith("лингвистика_")][:20]


### 2.1.3 Парсим конлю и приводим в нужный формат

In [ ]:
from conllu import parse

def parse_conllu(text): # берем предложение/текст и делаем словарь из словарей с признаками слова
    sentences = parse(text)
    parsed = []

    for sent in sentences:
        tokens = []
        for token in sent:
            sem_tags = extract_sem_tags(token.get("misc"))
            UD_lemma = conllu_to_w2v_format(token["lemma"], token["upos"])

            tokens.append({
                "form": token["form"],
                "lemma": token["lemma"],
                "upos": token["upos"],
                "UD_lemma": UD_lemma,
                "UD_pos": pos_map.get(token["upos"]),
                "feats": token["feats"],
                "head": token["head"],
                "deprel": token["deprel"],
                "sem": sem_tags
            })

        parsed.append(tokens)

    return parsed

# выдает набор токенов в таком формате:
# {'form': 'С', 'lemma': 'с', 'upos': 'ADP', 'UD_lemma': 'с_PR', 'UD_pos': 'PR', 'feats': None, 'head': 3, 'deprel': 'case', 'sem': []}


## 2.2 Загружаем конлю данные

In [ ]:
conllu_file = "/content/drive/MyDrive/Colab Notebooks/Аспирантура/conllu/handlabeled/GramEval2020-test-fiction.conllu"

with open(conllu_file, "r", encoding="utf-8") as f:
    conllu_data = f.read()


In [ ]:
conllu_text = parse_conllu(conllu_data)
# print(conllu_text)

for token in conllu_text[0]:
  print(token)


{'form': 'С', 'lemma': 'с', 'upos': 'ADP', 'UD_lemma': 'с_PR', 'UD_pos': 'PR', 'feats': None, 'head': 3, 'deprel': 'case', 'sem': []}
{'form': 'некоторых', 'lemma': 'некоторый', 'upos': 'DET', 'UD_lemma': 'некоторый_APRO', 'UD_pos': 'APRO', 'feats': {'Case': 'Gen', 'Number': 'Plur', 'PronType': 'Ind'}, 'head': 3, 'deprel': 'det', 'sem': []}
{'form': 'пор', 'lemma': 'пора', 'upos': 'NOUN', 'UD_lemma': 'пора_S', 'UD_pos': 'S', 'feats': {'Animacy': 'Inan', 'Case': 'Gen', 'Gender': 'Fem', 'Number': 'Plur'}, 'head': 5, 'deprel': 'obl', 'sem': ['abstr', 'time']}
{'form': 'я', 'lemma': 'я', 'upos': 'PRON', 'UD_lemma': 'я_SPRO', 'UD_pos': 'SPRO', 'feats': {'Case': 'Nom', 'Number': 'Sing', 'Person': '1', 'PronType': 'Prs'}, 'head': 5, 'deprel': 'nsubj', 'sem': ['concr', 'hum']}
{'form': 'полюбил', 'lemma': 'полюбить', 'upos': 'VERB', 'UD_lemma': 'полюбить_V', 'UD_pos': 'V', 'feats': {'Aspect': 'Perf', 'Gender': 'Masc', 'Mood': 'Ind', 'Number': 'Sing', 'Tense': 'Past', 'Transit': 'Tran', 'VerbFo

## 2.3 Считаем вектор набора слов (в формате conllu): берем среднее векторов всех слов в предложении

In [ ]:
import numpy as np

# на вход
# [{'form': 'С', 'lemma': 'с', 'upos': 'ADP', 'UD_lemma': 'с_PR', 'UD_pos': 'PR', 'feats': None, 'head': 3, 'deprel': 'case', 'sem': []}, {'form': 'некоторых', 'lemma': 'некоторый', 'upos': 'DET', 'UD_lemma': 'некоторый_APRO', 'UD_pos': 'APRO', 'feats': {'Case': 'Gen', 'Number': 'Plur', 'PronType': 'Ind'}, 'head': 3, 'deprel': 'det', 'sem': []}, {'form': 'пор', 'lemma': 'пора', 'upos': 'NOUN', 'UD_lemma': 'пора_S', 'UD_pos': 'S', 'feats': {'Animacy': 'Inan', 'Case': 'Gen', 'Gender': 'Fem', 'Number': 'Plur'}, 'head': 5, 'deprel': 'obl', 'sem': ['abstr', 'time']}, {'form': 'я', 'lemma': 'я', 'upos': 'PRON', 'UD_lemma': 'я_SPRO', 'UD_pos': 'SPRO', 'feats': {'Case': 'Nom', 'Number': 'Sing', 'Person': '1', 'PronType': 'Prs'}, 'head': 5, 'deprel': 'nsubj', 'sem': ['concr', 'hum']}, {'form': 'полюбил', 'lemma': 'полюбить', 'upos': 'VERB', 'UD_lemma': 'полюбить_V', 'UD_pos': 'V', 'feats': {'Aspect': 'Perf', 'Gender': 'Masc', 'Mood': 'Ind', 'Number': 'Sing', 'Tense': 'Past', 'Transit': 'Tran', 'VerbForm': 'Fin', 'Voice': 'Act'}, 'head': 0, 'deprel': 'root', 'sem': []}, {'form': 'это', 'lemma': 'этот', 'upos': 'DET', 'UD_lemma': 'этот_APRO', 'UD_pos': 'APRO', 'feats': {'Case': 'Acc', 'Gender': 'Neut', 'Number': 'Sing', 'PronType': 'Dem'}, 'head': 8, 'deprel': 'det', 'sem': []}, {'form': 'небольшое', 'lemma': 'небольшой', 'upos': 'ADJ', 'UD_lemma': 'небольшой_A', 'UD_pos': 'A', 'feats': {'Case': 'Acc', 'Degree': 'Pos', 'Gender': 'Neut', 'Number': 'Sing'}, 'head': 8, 'deprel': 'amod', 'sem': ['min', 'physq', 'size']}, {'form': 'кафе', 'lemma': 'кафе', 'upos': 'NOUN', 'UD_lemma': 'кафе_S', 'UD_pos': 'S', 'feats': {'Animacy': 'Inan', 'Case': 'Acc', 'Gender': 'Neut', 'InflClass': 'Ind', 'Number': 'Sing'}, 'head': 5, 'deprel': 'obj', 'sem': ['concr', 'org']}, {'form': '.', 'lemma': '.', 'upos': 'PUNCT', 'UD_lemma': '.', 'UD_pos': None, 'feats': None, 'head': 5, 'deprel': 'punct', 'sem': []}]
def conllu_text_to_vector(parsed_text, model):
    vectors = []

    for token in parsed_text:
        # Пропускаем пунктуацию и символы
        if token["upos"] in ["SYM", "PUNCT"]:
            continue

        # Проверяем наличие токена в Word2Vec
        if token["UD_lemma"] in model.wv.key_to_index:
            vectors.append(
                model.wv[token["UD_lemma"]]
            )

    # Если ни одного слова нет в модели
    if not vectors:
        return None

    return np.mean(vectors, axis=0)

    """
     Возвращает:
        np.ndarray размерности model.vector_size
        или None, если ни одного токена нет в словаре модели.
    """


In [ ]:
print(conllu_text[0])


[{'form': 'С', 'lemma': 'с', 'upos': 'ADP', 'UD_lemma': 'с_PR', 'UD_pos': 'PR', 'feats': None, 'head': 3, 'deprel': 'case', 'sem': []}, {'form': 'некоторых', 'lemma': 'некоторый', 'upos': 'DET', 'UD_lemma': 'некоторый_APRO', 'UD_pos': 'APRO', 'feats': {'Case': 'Gen', 'Number': 'Plur', 'PronType': 'Ind'}, 'head': 3, 'deprel': 'det', 'sem': []}, {'form': 'пор', 'lemma': 'пора', 'upos': 'NOUN', 'UD_lemma': 'пора_S', 'UD_pos': 'S', 'feats': {'Animacy': 'Inan', 'Case': 'Gen', 'Gender': 'Fem', 'Number': 'Plur'}, 'head': 5, 'deprel': 'obl', 'sem': ['abstr', 'time']}, {'form': 'я', 'lemma': 'я', 'upos': 'PRON', 'UD_lemma': 'я_SPRO', 'UD_pos': 'SPRO', 'feats': {'Case': 'Nom', 'Number': 'Sing', 'Person': '1', 'PronType': 'Prs'}, 'head': 5, 'deprel': 'nsubj', 'sem': ['concr', 'hum']}, {'form': 'полюбил', 'lemma': 'полюбить', 'upos': 'VERB', 'UD_lemma': 'полюбить_V', 'UD_pos': 'V', 'feats': {'Aspect': 'Perf', 'Gender': 'Masc', 'Mood': 'Ind', 'Number': 'Sing', 'Tense': 'Past', 'Transit': 'Tran', 'V

In [ ]:
print(conllu_text_to_vector(conllu_text[0], model))


[-0.6801689  -1.2015027  -0.19599073  0.14279331 -0.10957261  0.3392653
  0.85339975  0.35705996 -0.1536385   1.0885826   0.3210691   0.3267144
 -0.14453894  0.9184055  -0.30948555 -0.22838214  0.73388225 -0.3415522
  0.10410409 -0.33596623  0.07014459 -0.06908488  0.10499914  0.67133254
 -0.697199    0.25934598 -0.57755005  0.11451762  0.27287474  0.39506635
 -0.4078315  -0.45091313  1.0461425   0.05634477 -0.763152    0.03534341
 -0.43016464 -0.08048762 -0.16142973  0.07010497  0.13747893 -0.04078525
 -0.6791397  -0.70081985  0.2391685   0.4253048  -0.24954034  0.2570228
 -0.23743528  0.11375079  0.6961887   0.20795372  0.02235295 -0.278064
  0.6980036  -1.0014744   0.11639759 -0.30131745  0.8617058  -0.4018384
  0.4117828  -0.994474    0.9302652  -0.51452595  0.25348058  0.28097767
  0.21634841  0.12501422  0.5862499  -0.40683898 -1.1215291   0.07137611
  0.01164763 -0.16608283 -0.8590715   0.5224768   0.554659    0.58943653
 -0.07969092 -1.163989   -0.10101117  0.39826328 -0.184487

## 2.3 (Дополнительно) Собираем все употребления тега

In [ ]:
def extract_examples(conllu_text, target_tags=None):
    sentences = parse(conllu_text)
    results = []

    if target_tags:
        target_tags = [t.strip().lower() for t in target_tags]

    for sent in sentences:
        sentence_text = sent.metadata.get("text", "")

        for token in sent:
            sem_tags = extract_sem_tags(token.get("misc"))
            sem_tags = [t.lower() for t in sem_tags]

            if not sem_tags:
                continue

            if target_tags:
                if not any(tag in sem_tags for tag in target_tags):
                    continue

            results.append({
                "form": token["form"],
                "lemma": token["lemma"],
                "upos": token["upos"],
                "sem": sem_tags,
                "sentence": sentence_text,
                "token_id": token["id"]
            })

    return results


In [ ]:
target_tags = ["tool"]

searching_for_tags = extract_examples(conllu_data, target_tags)
for found_word_with_tag in searching_for_tags:
  print(found_word_with_tag)


{'form': 'автомат', 'lemma': 'автомат', 'upos': 'NOUN', 'sem': ['concr', 'device', 'tool'], 'sentence': 'Может быть, потому, что музыкальный автомат грохотал здесь не так оглушительно, как везде.', 'token_id': 8}
{'form': 'столик', 'lemma': 'столик', 'upos': 'NOUN', 'sem': ['concr', 'furn', 'tool'], 'sentence': 'Я приходил сюда каждый вечер и садился за угловой столик, откуда, слава богу, не виден был экран телевизора.', 'token_id': 10}
{'form': 'экран', 'lemma': 'экран', 'upos': 'NOUN', 'sem': ['concr', 'device', 'part', 'tool'], 'sentence': 'Я приходил сюда каждый вечер и садился за угловой столик, откуда, слава богу, не виден был экран телевизора.', 'token_id': 20}
{'form': 'телевизора', 'lemma': 'телевизор', 'upos': 'NOUN', 'sem': ['concr', 'device', 'tool'], 'sentence': 'Я приходил сюда каждый вечер и садился за угловой столик, откуда, слава богу, не виден был экран телевизора.', 'token_id': 21}
{'form': 'дверь', 'lemma': 'дверь', 'upos': 'NOUN', 'sem': ['concr', 'constr', 'part',

Попробуем сначала с контекстом во все предложение

# 3 Подготавливаем дефиниции

## 3.1 Грузим словарь с дефинициями

In [ ]:
from datasets import load_dataset

dictionary_definitions = load_dataset("snagbreac/russian-reverse-dictionary-train-data") # датасет со словарными определениями


In [ ]:
print(len(dictionary_definitions['train']))


294007


In [ ]:
print(dictionary_definitions['train'][1000])
# формат словаря: {'word': 'адъютантский', 'definition': 'принадлежащий адъютанту', 'df': 'efremova'}


{'word': 'адъютантский', 'definition': 'принадлежащий адъютанту', 'df': 'efremova'}


In [ ]:
def get_definitions(word, dataset):
    """
    Возвращает все словарные определения слова.
    """

    definitions = []

    for item in dataset:
        if item["word"].lower() == word.lower():
            definitions.append(item["definition"])

    return definitions


In [ ]:
adjutant_definitions = get_definitions(
    "адъютантский",
    dictionary_definitions['train']
)

print(adjutant_definitions)


['соотносящийся по знач. с сущ.: адъютант, связанный с ним', 'свойственный адъютанту, характерный для него', 'принадлежащий адъютанту']


## 3.2 Формируем словарь со всеми определениями и векторами определений для слова

In [ ]:
# индексируем определения чтобы удобнее обращаться
#TODO выгрузить чтобы не надо было каждый раз индексировать

# вот тут я подготавливаю словарь: https://colab.research.google.com/drive/1ue2LkofPbOfsDeEOg_h0vTR_Okb6BHBP?usp=drive_open#scrollTo=C1zJSK481ocy


In [ ]:
# from collections import defaultdict

# получаем словарь который сформировали в другом файле
# with open(OUTPUT_PATH, "rb") as f:
#     word2vectorized_def = pickle.load(f)


In [ ]:
"""
import numpy as np
import stanza

# Загружаем русский морфологический анализатор
#TODO: поменять на модель НКРЯ
stanza.download("ru")
nlp = stanza.Pipeline(
    lang="ru",
    processors="tokenize,pos,lemma",
    use_gpu=False
)
"""


'\nimport numpy as np\nimport stanza\n\n# Загружаем русский морфологический анализатор\n#TODO: поменять на модель НКРЯ\nstanza.download("ru")\nnlp = stanza.Pipeline(\n    lang="ru",\n    processors="tokenize,pos,lemma",\n    use_gpu=False\n)\n'



---


Что тут делаем:
- проходим по предложению
- токенизируем
- берем по слову из определения
  - с помощью stanza лемматизируем и получаем pos
  - кладем в словарь в формате ['слово_POS']

In [ ]:
!pip install razdel natasha


In [ ]:
from natasha import (
    Segmenter,
    MorphVocab,
    NewsEmbedding,
    NewsMorphTagger,
    Doc
)
# Инициализация Natasha
segmenter = Segmenter()
morph_vocab = MorphVocab()

emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)


In [ ]:
# функция чтобы считать средний вектор входного предложения (не в формате conllu)
def tokens_to_vector(text, model): # на вход словарь в формате ['слово_POS']

    doc = Doc(text)

    # Токенизация
    doc.segment(segmenter)

    # POS
    doc.tag_morph(morph_tagger)

    list_of_token_POS = []

    for token in doc.tokens:

        # Лемма
        token.lemmatize(morph_vocab)

        lemma = token.lemma
        pos = token.pos

        # Пропускаем пунктуацию и неизвестные POS
        if pos in ["PUNCT", "SYM"]:
            continue

        w2v_pos = pos_map.get(pos)

        if w2v_pos is None:
            continue

        list_of_token_POS.append(f"{lemma.lower()}_{w2v_pos}")


    vectors = []

    for word in list_of_token_POS:
      if word in model.wv.key_to_index:
            vectors.append(
                model.wv[word]
            )

    # Если ни одного слова нет в модели
    if not vectors:
        return None

    return np.mean(vectors, axis=0)

    """
     Возвращает:
        np.ndarray размерности model.vector_size
        или None, если ни одного токена нет в словаре модели.
    """


In [ ]:
print(tokens_to_vector(dictionary_definitions['train'][1000]['definition'], model))


[-2.0647231e-01 -1.1459212e+00  2.7519286e-02  1.0099009e+00
 -2.0873165e+00  4.9591288e-01  1.2051165e+00 -1.3655555e+00
  1.4837501e+00 -4.4303399e-01  3.0491173e-01 -1.2406504e+00
 -7.6597437e-02 -1.1647263e+00  1.5006348e+00  1.2592472e+00
 -1.6835020e+00  4.4443741e-01  2.6800543e-01  9.1133678e-01
  1.1459150e+00  4.4055629e-01 -6.9834614e-01 -1.1150208e+00
  1.4283484e+00 -5.4366875e-01 -4.6367359e-01  2.0021625e+00
  8.1510484e-01 -1.3902530e-01 -1.6484846e+00  1.3355639e+00
 -7.3161268e-01  1.3312523e+00  5.8702111e-01 -4.2176408e-01
  3.1958327e-02  1.2639136e+00  5.6945455e-01 -7.6599669e-01
 -1.4355066e-01 -4.5381218e-01  7.8639984e-02  5.3116345e-01
  1.8319103e-01 -5.8853149e-01 -4.3150955e-01  6.8354082e-01
  1.2631270e+00  1.1293478e+00  8.9137346e-02  1.0112653e+00
 -2.2030850e-01 -4.0965503e-01  6.9951755e-01 -1.0223978e+00
  7.8831196e-02  4.7588062e-01  2.0970309e-01 -4.0390521e-01
 -5.5986506e-01 -6.5879011e-01 -1.4413781e+00 -5.8832251e-02
 -2.5258139e-01 -5.06217

### делаем словарь с векторами дефиниций

In [ ]:
# def build_word2vectorized_def(dataset, model):

#     word2vectorized_def = {}
#     definition_vector_cache = {}

#     definition_id = 0

#     total = len(dataset)

#     for i, item in enumerate(dataset, start=1):

#         word = item["word"].lower()
#         definition = item["definition"]
#         dictionary_name = item.get("df")

#         if definition in definition_vector_cache:

#             definition_vector = definition_vector_cache[definition]

#         else:

#             definition_vector = tokens_to_vector(
#                 definition,
#                 model
#             )

#             definition_vector_cache[definition] = definition_vector

#         definition_id += 1

#         definition_data = {
#             "id": definition_id,
#             "dict": dictionary_name,
#             "definition": definition,
#             "vector": definition_vector
#         }

#         if word not in word2vectorized_def:
#             word2vectorized_def[word] = []

#         word2vectorized_def[word].append(definition_data)

#         if i % 1000 == 0:
#             print(f"Обработано: {i} / {total}")

#     return word2vectorized_def


проверяем на кусочке датасета:

In [ ]:
# import time

# test_data = dictionary_definitions["train"].select(range(1000))
# start = time.time()

# test_vectors = build_word2vectorized_def(
#     test_data,
#     model
# )

# elapsed = time.time() - start

# print(f"Время на 1000: {elapsed:.2f} сек")
# print(f"Скорость: {1000 / elapsed:.2f} деф./сек")

# estimated = 294007 / (1000 / elapsed)

# print(f"Оценочное время на 294007:")
# print(f"{estimated / 60:.1f} мин")


In [ ]:
# for i in test_vectors['абсолютистка']:
#   print(i)


In [ ]:
# word2vectorized_def = build_word2vectorized_def(
#     dictionary_definitions["train"],
#     model
# )


In [ ]:
# import pickle

# with open("word2vectorized_def.pkl", "wb") as f:
#     pickle.dump(word2vectorized_def, f)


In [ ]:
import pickle

with open("/content/drive/MyDrive/Colab Notebooks/Аспирантура/word2vectorized_def.pkl", "rb") as f:
    word2vectorized_def = pickle.load(f)

print(type(word2vectorized_def))
print(word2vectorized_def['адъютант'])


<class 'dict'>
[{'id': 992, 'dict': 'efremova', 'definition': 'должность офицера, состоящего при военном начальнике для выполнения различных поручений', 'vector': array([-2.22868845e-01, -3.00077289e-01, -9.36598182e-01, -2.49060869e-01,
       -5.23989081e-01,  2.10401297e-01, -6.46260232e-02,  8.48026294e-03,
        3.43782812e-01, -2.18122721e-01, -1.67292520e-01,  1.12759039e-01,
        8.83934647e-02, -5.65207958e-01,  2.96667427e-01,  5.06490588e-01,
        3.40828411e-02, -3.93443435e-01, -1.03283718e-01,  2.02508882e-01,
       -1.04178083e+00, -1.76814765e-01, -3.55696261e-01, -1.31544983e+00,
        1.08737528e-01, -9.46076512e-01, -1.18041515e-01, -2.67466933e-01,
        3.23276013e-01,  8.33720684e-01, -3.46237794e-02,  1.07618618e+00,
        3.79683673e-02, -9.71044526e-02,  3.37911189e-01,  1.25948942e+00,
        2.08318621e-01, -1.04393601e-01,  5.40486693e-01,  5.30097306e-01,
       -4.48246807e-01, -5.36210611e-02,  1.32885814e-01,  3.45779419e-01,
       -9.55

## 3.4 Получение наиболее близкой дефиниции

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
def cosine_sim(vec1, vec2):
    vec1 = np.asarray(vec1).reshape(1, -1)
    vec2 = np.asarray(vec2).reshape(1, -1)

    return float(cosine_similarity(vec1, vec2)[0][0])


In [ ]:
adjutant_text = adjutant_definitions[0]
print(adjutant_text)

print(cosine_sim(tokens_to_vector(adjutant_text, model), model.wv['человек_S']))
print(cosine_sim(tokens_to_vector(adjutant_text, model), model.wv['адъютант_S']))
print(cosine_sim(tokens_to_vector(adjutant_text, model), model.wv['профессия_S']))


соотносящийся по знач. с сущ.: адъютант, связанный с ним
0.0538092665374279
0.23522157967090607
-0.05649947375059128


In [ ]:
# для слова считаем вектор его контекста, грузим сюда; получаем векторы определений, ищем наиболее близкое к контексту
def find_best_definition(context_vector, list_of_definitions):
    """
    Ранжирует определения по семантической близости к контексту.

    context_vector:
        вектор слова в контексте

    list_of_definitions:
        список определений с векторами
    """

    if context_vector is None or not list_of_definitions:
        return []

    ranked_definitions = []

    for definition in list_of_definitions:
        definition_vector = definition['vector']

        if definition_vector is None:
            continue

        similarity = cosine_sim(
            context_vector,
            definition_vector
        )

        ranked_definitions.append({
            "definition": definition,
            "vector": definition_vector,
            "similarity": similarity
        })

    ranked_definitions.sort(
        key=lambda x: x["similarity"],
        reverse=True
    )

    return ranked_definitions


In [ ]:
# print(conllu_text[0])

# vector_predlizheniya_tselikom = conllu_text_to_vector(conllu_text[0], model)
# # print(vector_predlizheniya_tselikom)

# slovo_polyubil = conllu_text[0][4]['lemma']
# print(slovo_polyubil)

# definitions_polyubil = get_definitions(slovo_polyubil, dictionary_definitions['train'])
# # print(definitions_polyubil)


In [ ]:
# polyubil_best_definitions = find_best_definition(vector_predlizheniya_tselikom, definitions_polyubil)
# for i in polyubil_best_definitions:
#   print(i['similarity'], i['definition'])


# 4 Подготавливаем теги (словарь, определения → эмбеддинги)

тут я буду делать словарь тегов и их векторов: https://colab.research.google.com/drive/1hwfzqqva6tHuA-HXEl25TEpj_zYehTMV

## 4.1 Создаем словарь тегов (тег → описание/толкование тега)

In [ ]:
TAGS = {
    "abstr": "абстрактные понятия, не имеющие физической формы",
    "age": "возраст и возрастные характеристики",
    "animal": "животные и живые существа животного мира",
    "appear": "появление, возникновение чего-либо",
    "artwork": "произведения искусства как объекты творчества",
    "art": "искусство как деятельность и культурная сфера",
    "be": "существование, бытие, факт наличия",
    "behav": "поведение и модели действий",
    "berry": "ягоды как биологический класс плодов",
    "changest": "изменение состояния или характеристик",
    "cin": "кинематограф, кино как индустрия и искусство",
    "class": "класс, категория или группа объектов",
    "cloth": "одежда и текстильные изделия",
    "color": "цветовые характеристики и восприятие цвета",
    "concr": "конкретные предметные объекты",
    "constr": "здания и строительные сооружения",
    "contact": "контактирование и взаимодействие через соприкосновение или связь",
    "degr": "степень, мера интенсивности или выраженности",
    "device": "высокотехнологичные устройства и автоматические системы",
    "dir": "направление движения или ориентации",
    "disappear": "исчезновение или прекращение существования",
    "disease": "болезни и патологические состояния",
    "dish": "посуда и предметы кухонного использования",
    "dist": "расстояние между объектами",
    "doc": "документы и текстовые официальные материалы",
    "dram": "театр и драматическое искусство",
    "emot": "эмоции и эмоциональные состояния",
    "etn": "этнонимы и названия народов",
    "ev": "оценка, результат оценивания",
    "famn": "фамилии как имена семейного происхождения",
    "food": "еда и пищевые продукты",
    "form": "форма и структурные характеристики",
    "fruit": "плоды растений",
    "furn": "мебель и предметы обстановки",
    "grain": "зерновые культуры и зерно",
    "group": "группы людей или объединения",
    "hum": "человек как биологический и социальный субъект",
    "humq": "качества человека и личностные характеристики",
    "impact": "воздействие одного объекта на другой",
    "instr": "инструменты как средства выполнения действий",
    "inter": "взаимодействие между объектами или субъектами",
    "kin": "родственные отношения",
    "light": "свет и световые явления",
    "max": "большая степень, максимум значения",
    "ment": "ментальная и когнитивная сфера",
    "min": "малое значение, минимум",
    "move": "передвижение и движение",
    "mus": "музыка как искусство и звуковая система",
    "neg": "негативная оценка или отрицательная характеристика",
    "org": "организации и институциональные структуры",
    "part": "часть целого",
    "patrn": "отчество как часть имени",
    "perc": "восприятие и перцептивные процессы",
    "physiol": "физиологические процессы живых организмов",
    "physq": "физические свойства объектов",
    "place": "место и пространственные объекты",
    "plant": "растения и ботанические объекты",
    "posit": "позитивная оценка или положительная характеристика",
    "poss": "посессивная сфера, принадлежность",
    "prof": "профессии и виды профессиональной деятельности",
    "propn": "имена собственные",
    "psych": "психика и психические процессы",
    "put": "помещения и локализованные пространства",
    "qtm": "кванты и дискретные единицы",
    "quant": "количество и количественные характеристики",
    "set": "множества и совокупности объектов",
    "size": "размер и пространственные характеристики",
    "sound": "звук и акустические явления",
    "speech": "речь и речевая деятельность",
    "speed": "скорость движения или изменения",
    "stuff": "вещества и материалы",
    "taste": "вкус как сенсорная характеристика",
    "temper": "температура и тепловые характеристики",
    "text": "текстовые объекты и письменные данные",
    "time": "время и временные характеристики",
    "tool": "инструменты и орудия труда",
    "topon": "топонимы и географические названия",
    "transp": "транспорт и средства передвижения",
    "unit": "единицы измерения",
    "weapon": "оружие и военные средства",
    "weather": "погодные явления",
    "weight": "вес и масса объектов"
}


In [ ]:
tag_names = list(TAGS.keys())
tag_texts = list(TAGS.values())

# тоже сделать словарь {tag : {ru_name='название тега', ru_name_vector=[вектор названия]}}


## 4.2 Переводим описания тегов в векторы

In [ ]:
def descriptions_to_tag_vectors(tag_descriptions, model):
    """
    tag_descriptions:
        {
            "человек": "лицо, человек как носитель определённых признаков",
            "животное": "живое существо, не относящееся к человеку"
        }
    """

    tag2vector = {}

    for tag, description in tag_descriptions.items():

        result = tokens_to_vector(
            description,
            model
        )

        tag2vector[tag] = {
            "description": description,
            "vector": result
        }

    return tag2vector


In [ ]:
tag2vector = descriptions_to_tag_vectors(
    TAGS,
    model
)


In [ ]:
print(tag2vector['tool'])


{'description': 'инструменты и орудия труда', 'vector': array([-6.0046166e-01,  5.2704632e-02, -5.8456886e-01,  4.6149117e-01,
       -1.0282209e+00, -7.2873318e-01,  8.1017286e-01, -1.5107989e-03,
       -6.6207331e-01, -2.3183282e-01,  2.0654026e-01,  1.4661286e+00,
       -1.2564270e+00,  4.0470755e-01,  8.0107689e-02,  2.4314690e-01,
        1.5266612e-01, -5.3796941e-01, -1.4280223e+00,  4.8972398e-01,
       -1.6580362e+00,  9.4181985e-02,  6.1638170e-01, -1.5836880e-01,
        8.3448094e-01, -1.6363961e-01,  2.8279461e-03, -3.1890732e-01,
       -4.7662216e-01,  6.2639344e-01, -2.9841590e-01, -1.0855442e+00,
        3.0036065e-01,  3.0738556e-01,  2.3204222e-01,  6.0128009e-01,
        2.2560611e+00, -1.5976787e-02,  2.6801854e-01, -5.8735955e-01,
        3.3241224e-01, -1.7157449e-01,  7.8524387e-01, -4.9069542e-02,
        1.8564837e-01,  1.5238062e-01, -1.2999434e+00, -1.7725420e-01,
       -2.4257336e+00, -8.1739444e-01,  7.5327933e-01,  1.5384171e+00,
        5.8252805e-01

### 4.3 Предсказание семантических тегов

In [ ]:
def predict_tags(
    definition_vector,
    tag2vector,
    threshold=0.5
):
    """
    Сравнивает вектор дефиниции
    с векторами всех семантических тегов.
    """

    if definition_vector is None:
        return [], {}

    similarities = {}

    for tag, tag_data in tag2vector.items():

        # tag2vector хранится как:
        # {
        #     "tag": {
        #         "description": "...",
        #         "vector": [...]
        #     }
        # }

        tag_vector = tag_data["vector"]

        similarity = cosine_sim(
            definition_vector,
            tag_vector
        )

        similarities[tag] = similarity

    predicted_tags = [
        tag
        for tag, similarity in similarities.items()
        if similarity >= threshold
    ]

    return predicted_tags, similarities


In [ ]:
# print(polyubil_best_definitions[0]['definition'])


In [ ]:
# print(predict_tags(polyubil_best_definitions[0]['vector'], tag2vector, threshold=0.5))


# 5 Собираем все вместе

### 5.1. Значимые части речи


In [ ]:
SIGNIFICANT_UPOS = {
    "NOUN",
    "VERB",
    "ADJ",
    "ADV",
    "PROPN",
    "NUM",
    "PRON"
}


def is_significant_token(token):
    return token.get("upos") in SIGNIFICANT_UPOS


### 5.2 Обработка одного предложения

In [ ]:
def get_context_window(sentence_tokens, target_index, window_size=3):
    """
    Возвращает контекст вокруг целевого токена:
    window_size токенов слева и window_size токенов справа.

    Сам целевой токен в контекст не включается.
    """

    start = max(0, target_index - window_size)
    end = min(len(sentence_tokens), target_index + window_size + 1)

    context_tokens = (
        sentence_tokens[start:target_index]
        + sentence_tokens[target_index + 1:end]
    )

    return context_tokens


: 

In [ ]:
def process_sentence(
    sentence_tokens,
    model,
    word2vectorized_def,
    tag2vector,
    threshold=0.5
):
    """
    Обрабатывает одно предложение CoNLL-U.

    Для каждого значимого слова:
    1. считает вектор контекста;
    2. получает все дефиниции леммы;
    3. выбирает наиболее близкую дефиницию;
    4. сравнивает её с векторами семантических тегов;
    5. сохраняет результат.
    """

    results = []

     # --------------------------------------------------------
    # Обрабатываем токены
    # --------------------------------------------------------

    for target_index, token in enumerate(sentence_tokens):

        # Пропускаем незначимые части речи
        if not is_significant_token(token):
            continue

        form = token.get("form")
        lemma = token.get("lemma")
        gold_tags = token.get("sem", [])

        # ----------------------------------------------------
        # Получаем локальный контекст ±3 слова
        # ----------------------------------------------------

        context_tokens = get_context_window(
            sentence_tokens,
            target_index,
            window_size=3
        )

        # ----------------------------------------------------
        # Считаем вектор локального контекста
        # ----------------------------------------------------

        context_vector = conllu_text_to_vector(
            context_tokens,
            model
        )
        
         # Если контекст невозможно представить вектором
        if context_vector is None:
            results.append({
                "form": form,
                "lemma": lemma,
                "upos": token.get("upos"),
                "context_window": [
                    t.get("form")
                    for t in context_tokens
                ],
                "gold_tags": gold_tags,
                "definition": None,
                "definition_similarity": None,
                "predicted_tags": [],
                "tag_similarities": {},
                "status": "no_context_vector"
            })

            continue


        # ----------------------------------------------------
        # Получаем определения конкретной леммы
        # ----------------------------------------------------

        list_of_definitions = word2vectorized_def.get(
            lemma.lower(),
            []
        )

        # ----------------------------------------------------
        # Если определений нет
        # ----------------------------------------------------

        if not list_of_definitions:

            results.append({
                "form": form,
                "lemma": lemma,
                "upos": token.get("upos"),
                "gold_tags": gold_tags,
                "definition": None,
                "definition_similarity": None,
                "predicted_tags": [],
                "tag_similarities": {},
                "status": "no_definition"
            })

            continue

        # ----------------------------------------------------
        # Ранжируем определения
        # ----------------------------------------------------

        ranked_definitions = find_best_definition(
            context_vector,
            list_of_definitions
        )

        if not ranked_definitions:

            results.append({
                "form": form,
                "lemma": lemma,
                "upos": token.get("upos"),
                "gold_tags": gold_tags,
                "definition": None,
                "definition_similarity": None,
                "predicted_tags": [],
                "tag_similarities": {},
                "status": "no_definition_vector"
            })

            continue

        # ----------------------------------------------------
        # Берём наиболее близкую дефиницию
        # ----------------------------------------------------

        best_definition = ranked_definitions[0]

        definition_vector = best_definition["vector"]

        # ----------------------------------------------------
        # Предсказываем семантические теги
        # ----------------------------------------------------

        predicted_tags, tag_similarities = predict_tags(
            definition_vector,
            tag2vector,
            threshold=threshold
        )

        # ----------------------------------------------------
        # Сохраняем результат
        # ----------------------------------------------------

        results.append({
            "form": form,
            "lemma": lemma,
            "upos": token.get("upos"),
            "gold_tags": gold_tags,

            "definition": best_definition["definition"],
            "definition_similarity": best_definition["similarity"],

            "predicted_tags": predicted_tags,
            "tag_similarities": tag_similarities,

            "status": "classified"
        })

    return results


### 5.6. Обработка всего CoNLL-U

In [ ]:
def process_corpus(
    conllu_text,
    model,
    word2vectorized_def,
    tag2vector,
    threshold=0.5
):
    """
    Обрабатывает весь разобранный CoNLL-U.

    conllu_text:
        результат parse_conllu(conllu_data)

    model:
        Word2Vec-модель НКРЯ

    word2vectorized_def:
        словарь:
        {
            "слово": [
                {
                    "definition": "...",
                    "vector": [...]
                }
            ]
        }

    tag2vector:
        словарь:
        {
            "tag": {
                "description": "...",
                "vector": [...]
            }
        }
    """

    all_results = []

    for sentence_tokens in conllu_text:

        # ----------------------------------------------------
        # Восстанавливаем текст предложения
        # ----------------------------------------------------

        context = " ".join(
            token["form"]
            for token in sentence_tokens
        )

        # ----------------------------------------------------
        # Обрабатываем предложение
        # ----------------------------------------------------

        sentence_results = process_sentence(
            sentence_tokens=sentence_tokens,
            model=model,
            word2vectorized_def=word2vectorized_def,
            tag2vector=tag2vector,
            threshold=threshold
        )

        all_results.extend(sentence_results)

    return all_results


In [ ]:
results = process_corpus(
    conllu_text=conllu_text,
    model=model,
    word2vectorized_def=word2vectorized_def,
    tag2vector=tag2vector,
    threshold=0.5
)


# 6 Оцениваем!!! Наконец-то блин

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    hamming_loss
)

def evaluate_multilabel(results):

    classified = [
        r for r in results
        if r["status"] == "classified"
    ]

    y_true = [
        r["gold_tags"]
        for r in classified
    ]

    y_pred = [
        r["predicted_tags"]
        for r in classified
    ]

    mlb = MultiLabelBinarizer()

    y_true_bin = mlb.fit_transform(y_true)
    y_pred_bin = mlb.transform(y_pred)

    metrics = {
        "micro_precision": precision_score(
            y_true_bin,
            y_pred_bin,
            average="micro",
            zero_division=0
        ),

        "micro_recall": recall_score(
            y_true_bin,
            y_pred_bin,
            average="micro",
            zero_division=0
        ),

        "micro_f1": f1_score(
            y_true_bin,
            y_pred_bin,
            average="micro",
            zero_division=0
        ),

        "macro_precision": precision_score(
            y_true_bin,
            y_pred_bin,
            average="macro",
            zero_division=0
        ),

        "macro_recall": recall_score(
            y_true_bin,
            y_pred_bin,
            average="macro",
            zero_division=0
        ),

        "macro_f1": f1_score(
            y_true_bin,
            y_pred_bin,
            average="macro",
            zero_division=0
        ),

        "exact_match": accuracy_score(
            y_true_bin,
            y_pred_bin
        ),

        "hamming_loss": hamming_loss(
            y_true_bin,
            y_pred_bin
        )
    }

    return metrics, mlb


In [ ]:
metrics, mlb = evaluate_multilabel(results)

for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")


micro_precision: 0.1014
micro_recall: 0.1171
micro_f1: 0.1087
macro_precision: 0.0967
macro_recall: 0.0861
macro_f1: 0.0748
exact_match: 0.1802
hamming_loss: 0.0472


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['age', 'art', 'artwork', 'color', 'contact', 'dir', 'disease', 'dist', 'dram', 'form', 'fruit', 'grain', 'light', 'put', 'taste', 'temper', 'topon', 'weapon', 'weather', 'weight'] will be ignored
  warnings.warn(


считаем количества и долю классов

In [ ]:
from collections import Counter

classified = [
        r for r in results
        if r["status"] == "classified"
]

y_true = [
        r["gold_tags"]
        for r in classified
]

y_pred = [
        r["predicted_tags"]
        for r in classified
]

true_counts = Counter(
    label
    for labels in y_true
    for label in labels
)

# print("Распределение классов в y_true:")
# for label, count in true_counts.most_common():
#     print(f"{label}: {count}")

total_labels = sum(true_counts.values())

print("Распределение классов:")
for label, count in true_counts.most_common():
    percentage = count / total_labels * 100
    print(f"{label}: {count} ({percentage:.2f}%)")


Распределение классов:
concr: 175 (20.76%)
hum: 127 (15.07%)
abstr: 77 (9.13%)
time: 36 (4.27%)
part: 30 (3.56%)
move: 29 (3.44%)
speech: 25 (2.97%)
ment: 23 (2.73%)
emot: 23 (2.73%)
place: 22 (2.61%)
ev: 21 (2.49%)
tool: 15 (1.78%)
perc: 13 (1.54%)
posit: 12 (1.42%)
physiol: 12 (1.42%)
be: 11 (1.30%)
prof: 11 (1.30%)
quant: 10 (1.19%)
degr: 9 (1.07%)
psych: 8 (0.95%)
max: 8 (0.95%)
propn: 8 (0.95%)
min: 7 (0.83%)
stuff: 7 (0.83%)
humq: 7 (0.83%)
impact: 7 (0.83%)
physq: 6 (0.71%)
org: 6 (0.71%)
neg: 6 (0.71%)
changest: 6 (0.71%)
kin: 6 (0.71%)
set: 6 (0.71%)
device: 5 (0.59%)
furn: 5 (0.59%)
speed: 5 (0.59%)
food: 5 (0.59%)
class: 5 (0.59%)
appear: 5 (0.59%)
text: 5 (0.59%)
sound: 4 (0.47%)
poss: 4 (0.47%)
unit: 4 (0.47%)
plant: 3 (0.36%)
animal: 3 (0.36%)
qtm: 3 (0.36%)
size: 2 (0.24%)
inter: 2 (0.24%)
constr: 2 (0.24%)
etn: 2 (0.24%)
mus: 1 (0.12%)
famn: 1 (0.12%)
sport: 1 (0.12%)
instr: 1 (0.12%)
doc: 1 (0.12%)
transp: 1 (0.12%)
disappear: 1 (0.12%)
group: 1 (0.12%)
behav: 1 (0.12%

считаем метрики отдельно для каждого класса

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_recall_fscore_support

mlb = MultiLabelBinarizer()

y_true_bin = mlb.fit_transform(y_true)
y_pred_bin = mlb.transform(y_pred)

precision, recall, f1, support = precision_recall_fscore_support(
    y_true_bin,
    y_pred_bin,
    average=None,
    zero_division=0
)

for label, p, r, f, s in zip(
    mlb.classes_,
    precision,
    recall,
    f1,
    support
):
    print(
        f"{label:20s} "
        f"P={p:.4f} "
        f"R={r:.4f} "
        f"F1={f:.4f} "
        f"support={s}"
    )


abstr                P=0.1429 R=0.1039 F1=0.1203 support=77
animal               P=0.0227 R=0.3333 F1=0.0426 support=3
appear               P=0.0000 R=0.0000 F1=0.0000 support=5
be                   P=0.0000 R=0.0000 F1=0.0000 support=11
behav                P=0.0000 R=0.0000 F1=0.0000 support=1
changest             P=0.0000 R=0.0000 F1=0.0000 support=6
class                P=0.0000 R=0.0000 F1=0.0000 support=5
cloth                P=0.5000 R=1.0000 F1=0.6667 support=1
concr                P=0.0000 R=0.0000 F1=0.0000 support=175
constr               P=0.0000 R=0.0000 F1=0.0000 support=2
degr                 P=0.1111 R=0.1111 F1=0.1111 support=9
device               P=0.2000 R=0.2000 F1=0.2000 support=5
disappear            P=0.0000 R=0.0000 F1=0.0000 support=1
doc                  P=0.2000 R=1.0000 F1=0.3333 support=1
emot                 P=0.0227 R=0.0435 F1=0.0299 support=23
etn                  P=0.0000 R=0.0000 F1=0.0000 support=2
ev                   P=0.3333 R=0.0476 F1=0.0833 su

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['age', 'art', 'artwork', 'cin', 'color', 'contact', 'dir', 'disease', 'dish', 'dram', 'form', 'fruit', 'grain', 'light', 'put', 'taste', 'temper', 'topon', 'weapon', 'weather', 'weight'] will be ignored
  warnings.warn(


In [ ]:
import pandas as pd

results = pd.DataFrame({
    'class': mlb.classes_,
    'precision': precision,
    'recall': recall,
    'f1': f1,
    'support': support
})

results = results.sort_values(
    'support',
    ascending=False
)

print(results)


        class  precision    recall        f1  support
8       concr   0.000000  0.000000  0.000000      175
21        hum   0.661538  0.338583  0.447917      127
0       abstr   0.142857  0.103896  0.120301       77
55       time   0.500000  0.194444  0.280000       36
34       part   0.500000  0.066667  0.117647       30
30       move   0.166667  0.034483  0.057143       29
50     speech   0.193548  0.240000  0.214286       25
14       emot   0.022727  0.043478  0.029851       23
28       ment   0.066667  0.130435  0.088235       23
38      place   0.428571  0.272727  0.333333       22
16         ev   0.333333  0.047619  0.083333       21
56       tool   0.250000  0.066667  0.105263       15
35       perc   0.000000  0.000000  0.000000       13
36    physiol   0.187500  0.250000  0.214286       12
40      posit   0.083333  0.083333  0.083333       12
3          be   0.000000  0.000000  0.000000       11
42       prof   0.105263  0.181818  0.133333       11
46      quant   0.500000  0.

#